# PyBaMM SEI Formulations Sensitivity Analysis
**Last updated: 2026-04-04 13:25 (Local Time)**

This notebook simulates calendar ageing followed by fast cycling, designed to run ONE MODEL AT A TIME.
After each run, **restart the runtime** (`Runtime -> Restart session`) to fully free C++ solver memory before running the next model.

Available SEI options:
* `"reaction limited"`
* `"solvent-diffusion limited"`
* `"electron-migration limited"`
* `"interstitial-diffusion limited"`

In [ ]:
!pip install pybamm==26.3 pandas matplotlib

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

pybamm.set_logging_level("NOTICE")

## Configuration
**Change `CURRENT_MODEL` below, then Run All. Restart runtime between models.**

In [ ]:
CURRENT_MODEL = "reaction limited"  # <--- Change this string!

STORAGE_DAYS = 30
INITIAL_SOC = 0.9
TOTAL_CYCLES = 100
CHUNK_SIZE = 10
MIN_CAPACITY = 2.0  # A.h — stop simulation if total charge capacity drops below this

EXPERIMENT_STEP = (
    "Discharge at C/9 until 3.2 V",
    "Rest for 15 minutes",
    "Charge at C/2 until 4.1 V",
    "Hold at 4.1 V until C/20",
    "Rest for 15 minutes",
)

# SEI tracking variables
SEI_VARS = {
    "Neg SEI Thickness [m]": "X-averaged negative SEI thickness [m]",
    "Pos SEI Thickness [m]": "X-averaged positive SEI thickness [m]",
    "Neg SEI Capacity Loss [A.h]": "Loss of capacity to negative SEI [A.h]",
    "Pos SEI Capacity Loss [A.h]": "Loss of capacity to positive SEI [A.h]",
    "Neg SEI Cracks Thickness [m]": "X-averaged negative SEI on cracks thickness [m]",
    "Pos SEI Cracks Thickness [m]": "X-averaged positive SEI on cracks thickness [m]",
}

def get_submesh_types():
    model = pybamm.lithium_ion.DFN()
    return model.default_submesh_types.copy()

var_pts = {"x_n": 30, "x_s": 30, "x_p": 30, "r_n": 50, "r_p": 50}

## Execution
Runs 30-day storage ageing followed by chunked cycling, using `set_initial_conditions_from()` to avoid accumulating solution history in memory. Stops early if total capacity drops below 2 A.h.

In [ ]:
print(f"\n{'='*50}")
print(f"Running Isolated SEI Model: {CURRENT_MODEL}")
print(f"{'='*50}\n")

options = {
    "SEI": CURRENT_MODEL,
    "SEI porosity change": "true",
    "lithium plating": "partially reversible",
    "lithium plating porosity change": "true",
    "particle mechanics": ("swelling and cracking", "swelling only"),
    "SEI on cracks": "true",
    "loss of active material": "stress-driven",
}

parameter_values = pybamm.ParameterValues("OKane2022")
solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-6)

# --- Ageing Phase ---
print(f"-> Storage Phase ({STORAGE_DAYS} days)")
model_age = pybamm.lithium_ion.DFN(options)
parameter_values_age = pybamm.ParameterValues("OKane2022")
parameter_values_age["Current function [A]"] = 0

seconds = STORAGE_DAYS * 24 * 60 * 60
t_eval = np.linspace(0, seconds, min(100, max(20, STORAGE_DAYS * 2)))
sim_age = pybamm.Simulation(
    model_age, parameter_values=parameter_values_age, solver=solver,
    var_pts=var_pts, submesh_types=get_submesh_types()
)
sol_ageing = sim_age.solve(t_eval=t_eval, initial_soc=INITIAL_SOC)
print("   Storage complete.")

# --- Cycling Phase ---
print(f"-> Cycling Phase ({TOTAL_CYCLES} cycles, chunked by {CHUNK_SIZE})")
print(f"   Early stop threshold: total capacity < {MIN_CAPACITY} A.h")
num_chunks = TOTAL_CYCLES // CHUNK_SIZE
experiment_chunk = pybamm.Experiment([EXPERIMENT_STEP] * CHUNK_SIZE)

cc_caps, cv_caps, dis_caps, cycles = [], [], [], []
sei_data = {key: [] for key in SEI_VARS}
current_solution = sol_ageing
parameter_values_cyc = pybamm.ParameterValues("OKane2022")
capacity_exhausted = False

for chunk_idx in range(num_chunks):
    if capacity_exhausted:
        break

    start_cycle = chunk_idx * CHUNK_SIZE + 1
    print(f"   Chunk {chunk_idx + 1}/{num_chunks} (Cycles {start_cycle}-{start_cycle + CHUNK_SIZE - 1})... ", end="")

    model_cyc = pybamm.lithium_ion.DFN(options)

    sim_cyc = pybamm.Simulation(
        model_cyc,
        experiment=experiment_chunk,
        parameter_values=parameter_values_cyc,
        solver=solver,
        var_pts=var_pts,
        submesh_types=get_submesh_types(),
    )

    sim_cyc.model.set_initial_conditions_from(current_solution)

    try:
        sim_cyc.solve()
        sol = sim_cyc.solution

        for i, cycle_sol in enumerate(sol.cycles):
            current_cycle_num = start_cycle + i
            steps = cycle_sol.steps

            step_dis = steps[0] if len(steps) > 0 else None
            step_cc  = steps[2] if len(steps) > 2 else None
            step_cv  = steps[3] if len(steps) > 3 else None

            d_cap = abs(
                step_dis["Discharge capacity [A.h]"].entries[-1]
                - step_dis["Discharge capacity [A.h]"].entries[0]
            ) if step_dis else 0.0

            c_cap = abs(
                step_cc["Discharge capacity [A.h]"].entries[-1]
                - step_cc["Discharge capacity [A.h]"].entries[0]
            ) if step_cc else 0.0

            v_cap = abs(
                step_cv["Discharge capacity [A.h]"].entries[-1]
                - step_cv["Discharge capacity [A.h]"].entries[0]
            ) if step_cv else 0.0

            total_cap = c_cap + v_cap

            if total_cap < MIN_CAPACITY:
                print(f"\n   *** Capacity dropped to {total_cap:.3f} A.h at cycle {current_cycle_num}. Stopping. ***")
                capacity_exhausted = True
                break

            dis_caps.append(d_cap)
            cc_caps.append(c_cap)
            cv_caps.append(v_cap)
            cycles.append(current_cycle_num)

            # Extract SEI data at end of each cycle
            for col_name, pybamm_var in SEI_VARS.items():
                try:
                    val = cycle_sol[pybamm_var].entries[-1]
                    sei_data[col_name].append(float(val))
                except (KeyError, IndexError):
                    sei_data[col_name].append(0.0)

        current_solution = sim_cyc.solution
        if not capacity_exhausted:
            print("Done.")

    except Exception as e:
        print(f"STOPPED - {e}")
        break

    del sim_cyc
    gc.collect()

print(f"\nCompleted {len(cycles)} cycles total.")

## Analysis and Data Export

In [ ]:
rows = []
for idx, c in enumerate(cycles):
    row = {
        "Variant": CURRENT_MODEL,
        "Cycle": c,
        "CC Capacity [A.h]": cc_caps[idx],
        "CV Capacity [A.h]": cv_caps[idx],
        "CC/CV Ratio": cc_caps[idx] / cv_caps[idx] if cv_caps[idx] > 0 else float('nan')
    }
    for col_name in SEI_VARS:
        row[col_name] = sei_data[col_name][idx] if idx < len(sei_data[col_name]) else 0.0
    rows.append(row)

df = pd.DataFrame(rows)

milestones = [10, 50, 100, 150, 200]
df_milestones = df[df['Cycle'].isin(milestones)]
print(f"\n--- {CURRENT_MODEL} Milestone Table ---")
print(df_milestones[['Cycle', 'CC Capacity [A.h]', 'CV Capacity [A.h]', 'CC/CV Ratio']].to_string(index=False))

print(f"\n--- SEI Thickness at Milestones ---")
sei_cols = ['Cycle', 'Neg SEI Thickness [m]', 'Pos SEI Thickness [m]', 'Neg SEI Cracks Thickness [m]', 'Pos SEI Cracks Thickness [m]']
available_cols = [c for c in sei_cols if c in df_milestones.columns]
print(df_milestones[available_cols].to_string(index=False))

print(f"\n--- SEI Capacity Loss at Milestones ---")
loss_cols = ['Cycle', 'Neg SEI Capacity Loss [A.h]', 'Pos SEI Capacity Loss [A.h]']
available_cols = [c for c in loss_cols if c in df_milestones.columns]
print(df_milestones[available_cols].to_string(index=False))

filename = f"{CURRENT_MODEL.replace(' ', '_').replace('-', '_')}_data.csv"
df.to_csv(filename, index=False)
print(f"\nSaved {len(cycles)} cycles to: {filename}")
print("Download this CSV before restarting the runtime!")

## Visual Plots

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Capacity
axs[0, 0].plot(cycles, cc_caps, marker=".", label=CURRENT_MODEL, color='blue')
axs[0, 0].set_title("CC Capacity Decay")
axs[0, 0].set_ylabel("Capacity [A.h]")
axs[0, 0].set_xlabel("Cycles")
axs[0, 0].grid(True)
axs[0, 0].legend()

axs[0, 1].plot(cycles, cv_caps, marker=".", label=CURRENT_MODEL, color='orange')
axs[0, 1].set_title("CV Capacity Reliance")
axs[0, 1].set_ylabel("Capacity [A.h]")
axs[0, 1].set_xlabel("Cycles")
axs[0, 1].grid(True)
axs[0, 1].legend()

ratio = [cc / cv if cv > 0 else 0 for cc, cv in zip(cc_caps, cv_caps)]
axs[0, 2].plot(cycles, ratio, marker=".", label=CURRENT_MODEL, color='green')
axs[0, 2].set_title("CC/CV Ratio")
axs[0, 2].set_ylabel("Ratio")
axs[0, 2].set_xlabel("Cycles")
axs[0, 2].grid(True)
axs[0, 2].legend()

# Row 2: SEI
axs[1, 0].plot(cycles, sei_data["Neg SEI Thickness [m]"], marker=".", label="Negative", color='red')
axs[1, 0].plot(cycles, sei_data["Pos SEI Thickness [m]"], marker=".", label="Positive", color='purple')
axs[1, 0].set_title("SEI Film Thickness")
axs[1, 0].set_ylabel("Thickness [m]")
axs[1, 0].set_xlabel("Cycles")
axs[1, 0].grid(True)
axs[1, 0].legend()

axs[1, 1].plot(cycles, sei_data["Neg SEI Cracks Thickness [m]"], marker=".", label="Negative (cracks)", color='red', linestyle='--')
axs[1, 1].plot(cycles, sei_data["Pos SEI Cracks Thickness [m]"], marker=".", label="Positive (cracks)", color='purple', linestyle='--')
axs[1, 1].set_title("SEI on Cracks Thickness")
axs[1, 1].set_ylabel("Thickness [m]")
axs[1, 1].set_xlabel("Cycles")
axs[1, 1].grid(True)
axs[1, 1].legend()

axs[1, 2].plot(cycles, sei_data["Neg SEI Capacity Loss [A.h]"], marker=".", label="Negative Loss", color='red')
axs[1, 2].plot(cycles, sei_data["Pos SEI Capacity Loss [A.h]"], marker=".", label="Positive Loss", color='purple')
axs[1, 2].set_title("Capacity Lost to SEI")
axs[1, 2].set_ylabel("Capacity Loss [A.h]")
axs[1, 2].set_xlabel("Cycles")
axs[1, 2].grid(True)
axs[1, 2].legend()

plt.suptitle(f"SEI Model: {CURRENT_MODEL}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()